### Quality check analysis of anndata object from re-mapped E-MTAB-≈ samples
- **Developed by:** Anna Maguza
- **Affilation:** Faculty of Medicine, Würzburg University
- **Creation date:** 9th of December 2024
- **Last modified date:** 9th of December 2024

This notebook contains such parts:
* Doublets score prediction with `scrublet`
* Calculating quality metrics with `sctk`
* Identify cells that pass QC on:
    * Individual cell level
    * QC cluster level
    * Both individual cell and cluster levels
* Visualize quality metrics before filtering
* Filter out cells that do not pass QC
* Visualize quality metrics after filtering
* Identify sex covariates
* Calculate cell cycle score

### Import packages

In [1]:
import scanpy as sc
import scrublet as scr
import numpy as np
import pandas as pd
import sctk
import muon as mu
from datetime import datetime
import os
import json
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
import anndata as ad

2024-12-09 10:32:12.451126: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-12-09 10:32:12.451167: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-12-09 10:32:12.451191: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-09 10:32:13.097543: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


### Set up parameters to save plots and objects

In [ ]:
project = 'gut'
species = 'hs'
atribute = 'Holloway2021_E-MTAB-9720'
name = 'AM'
timestamp = datetime.now().strftime('%d%m%Y_%H%M%S')
counts = 'raw'

### Load data

In [ ]:
adata = sc.read_h5ad('Holloway_2021/gut_hs_Holloway2021_E-MTAB-9720_QC_AM_27112024_160858_raw.h5ad')
adata

AnnData object with n_obs × n_vars = 106706 × 70711
    obs: 'sample_name', 'batch', 'barcode', 'Source Name', 'Comment[ENA_SAMPLE]', 'Comment[BioSD_SAMPLE]', 'Characteristics[organism]', 'Characteristics[disease]', 'Characteristics[genotype]', 'Characteristics[organism part]', 'Characteristics[cell type]', 'Characteristics[growth condition]', 'Characteristics[developmental stage]', 'Material Type', 'Protocol REF', 'Protocol REF.1', 'Protocol REF.2', 'Protocol REF.3', 'Extract Name', 'Comment[LIBRARY_LAYOUT]', 'Comment[LIBRARY_SELECTION]', 'Comment[LIBRARY_SOURCE]', 'Comment[LIBRARY_STRATEGY]', 'Comment[cdna read]', 'Comment[cdna read offset]', 'Comment[cdna read size]', 'Comment[cell barcode offset]', 'Comment[cell barcode read]', 'Comment[cell barcode size]', 'Comment[end bias]', 'Comment[input molecule]', 'Comment[library construction]', 'Comment[primer]', 'Comment[sample barcode offset]', 'Comment[sample barcode read]', 'Comment[sample barcode size]', 'Comment[single cell isolation

## Basic filtering

In [5]:
sc.pp.filter_cells(adata, min_genes=100)
adata

AnnData object with n_obs × n_vars = 239919 × 70711
    obs: 'Extract Name', 'batch', 'barcode', 'Source Name', 'Comment[ENA_SAMPLE]', 'Comment[BioSD_SAMPLE]', 'Characteristics[organism]', 'Characteristics[age]', 'Unit[time unit]', 'Term Source REF', 'Term Accession Number', 'Characteristics[developmental stage]', 'Characteristics[sex]', 'Characteristics[individual]', 'Characteristics[cell type]', 'Characteristics[organism part]', 'Material Type', 'Protocol REF', 'Performer', 'Protocol REF.1', 'Performer.1', 'Protocol REF.2', 'Performer.2', 'Comment[LIBRARY_LAYOUT]', 'Comment[LIBRARY_SELECTION]', 'Comment[LIBRARY_SOURCE]', 'Comment[LIBRARY_STRAND]', 'Comment[LIBRARY_STRATEGY]', 'Comment[NOMINAL_LENGTH]', 'Comment[NOMINAL_SDEV]', 'Comment[ORIENTATION]', 'Comment[cdna read]', 'Comment[cdna read offset]', 'Comment[cdna read size]', 'Comment[cell barcode offset]', 'Comment[cell barcode read]', 'Comment[cell barcode size]', 'Comment[end bias]', 'Comment[input molecule]', 'Comment[library co

## Doublet score prediction

In [ ]:
sample_names = adata.obs['Source Name'].unique()

for sample_name in sample_names:
    mask = adata.obs['Source Name'] == sample_name
    sample_adata = adata[mask].copy()

    scrub = scr.Scrublet(sample_adata.X)

    sample_adata.obs['doublet_scores'], sample_adata.obs['predicted_doublets'] = scrub.scrub_doublets()

    adata.obs.loc[mask, 'doublet_scores'] = sample_adata.obs['doublet_scores']
    adata.obs.loc[mask, 'predicted_doublets'] = sample_adata.obs['predicted_doublets']

    #scrub.plot_histogram()

    #plt.show()

In [4]:
doub_tab = pd.crosstab(adata.obs['Source Name'],adata.obs['predicted_doublets'])
doub_tab.sum()

predicted_doublets
False    63330
None     43375
True         1
dtype: int64

In [5]:
doub_tab_percentage = doub_tab.div(doub_tab.sum(axis=1), axis=0) * 100

In [6]:
doub_tab_percentage

predicted_doublets,False,None,True
Source Name,,,
HT071-LWRN-EGF,100.000000,0.0,0.000000
HT071-LWRN-NRG1,99.992626,0.0,0.007374
HT323-LWRN-EGF,100.000000,0.0,0.000000
HT323-LWRN-EGF-NRG1,100.000000,0.0,0.000000
HT323-LWRN-NRG1,0.000000,100.0,0.000000


In [ ]:
plt.figure(figsize=(15, 7), dpi = 300)
sns.barplot(x=doub_tab_percentage.index, y=doub_tab_percentage[('True')])
plt.ylim(0, 100)
plt.xticks(rotation=90)
plt.xlabel('Source Name')
plt.ylabel('Percentage of Doublets')
plt.title('Percentage of Doublets per Sample')
plt.savefig(f"Holloway_2021/figures/QC/{atribute}_scrublet_doublets_{timestamp}.png", bbox_inches="tight")
plt.show()

## Quality Check

+ Calculate QC metrics

In [ ]:
sctk.calculate_qc(adata)

In [4]:
adata.obs['total_counts'] = adata.X.sum(axis=1)

In [5]:
adata.obs['n_genes_by_counts'] = (adata.X > 0).sum(axis=1)

+ Determine which of the cells pass QC individually

In [ ]:
sctk.cellwise_qc(adata)
adata

In [6]:
adata.obs['cell_passed_qc'].sum()

11554

In [12]:
adata.uns['scautoqc_ranges']

,low,high
n_counts,"{""n_counts"": 939.0557404113508, ""n_genes"": 99....","{""n_counts"": 189344.03125, ""n_genes"": 8746.997..."
n_genes,"{""n_counts"": 939.0557404113508, ""n_genes"": 99....","{""n_counts"": 189344.03125, ""n_genes"": 8746.997..."
percent_mito,"{""n_counts"": 939.0557404113508, ""n_genes"": 99....","{""n_counts"": 189344.03125, ""n_genes"": 8746.997..."
percent_ribo,"{""n_counts"": 939.0557404113508, ""n_genes"": 99....","{""n_counts"": 189344.03125, ""n_genes"": 8746.997..."
percent_hb,"{""n_counts"": 939.0557404113508, ""n_genes"": 99....","{""n_counts"": 189344.03125, ""n_genes"": 8746.997..."


+ Check the QC parameters (default parameters)

In [14]:
sctk.default_metric_params_df

,min,max,scale,side,min_pass_rate
n_counts,1000.00,NaN,log,min_only,0.10
n_genes,100.00,NaN,log,min_only,0.10
percent_mito,0.01,20.0,log,max_only,0.10
percent_ribo,0.00,100.0,log,both,0.10
percent_hb,NaN,1.0,log,max_only,0.10
percent_soup,NaN,5.0,log,max_only,0.10
percent_spliced,50.00,97.5,log,both,0.10
scrublet_score,NaN,0.3,linear,max_only,0.95


+ Cluster cells based on their QC

In [21]:
metrics_list = ["log1p_n_counts", "log1p_n_genes", "percent_mito", "percent_ribo", "percent_hb"]
sctk.generate_qc_clusters(adata, metrics = metrics_list)
adata

AnnData object with n_obs × n_vars = 239919 × 70711
    obs: 'Extract Name', 'batch', 'barcode', 'Source Name', 'Comment[ENA_SAMPLE]', 'Comment[BioSD_SAMPLE]', 'Characteristics[organism]', 'Characteristics[age]', 'Unit[time unit]', 'Term Source REF', 'Term Accession Number', 'Characteristics[developmental stage]', 'Characteristics[sex]', 'Characteristics[individual]', 'Characteristics[cell type]', 'Characteristics[organism part]', 'Material Type', 'Protocol REF', 'Performer', 'Protocol REF.1', 'Performer.1', 'Protocol REF.2', 'Performer.2', 'Comment[LIBRARY_LAYOUT]', 'Comment[LIBRARY_SELECTION]', 'Comment[LIBRARY_SOURCE]', 'Comment[LIBRARY_STRAND]', 'Comment[LIBRARY_STRATEGY]', 'Comment[NOMINAL_LENGTH]', 'Comment[NOMINAL_SDEV]', 'Comment[ORIENTATION]', 'Comment[cdna read]', 'Comment[cdna read offset]', 'Comment[cdna read size]', 'Comment[cell barcode offset]', 'Comment[cell barcode read]', 'Comment[cell barcode size]', 'Comment[end bias]', 'Comment[input molecule]', 'Comment[library co

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=180, figsize=(10, 10))
    sc.pl.embedding(adata, "X_umap_qc", color=["log1p_n_counts", "log1p_n_genes",
                                            "percent_mito", "percent_ribo", "percent_hb", "qc_cluster"], 
                                            color_map="magma_r", frameon=False, ncols=3, size=1, show=False)
    plt.savefig(f"Holloway_2021/figures/QC/{atribute}_qc_{timestamp}.png", bbox_inches="tight")
    plt.show()

+ Determine clusters that pass QC

In [ ]:
sctk.clusterwise_qc(adata)
adata

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(15, 15))
    sc.pl.embedding(adata, "X_umap_qc", color=["cell_passed_qc", "cluster_passed_qc", 'Source Name'],frameon=False, ncols=3, size=5, show=False)
    plt.savefig(f"Holloway_2021/figures/QC/{atribute}_sctk_qc_{timestamp}.png", bbox_inches="tight")
    plt.show()

+ Find the consensus between clusters that pass QC and individual cells that pass QC

In [ ]:
sctk.multi_resolution_cluster_qc(adata, metrics = metrics_list)

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(15, 15))
    sc.pl.embedding(adata, "X_umap_qc", color=["cell_passed_qc", 
                                           "cluster_passed_qc", 
                                           "consensus_fraction", 
                                           "consensus_passed_qc", 'Source Name'],frameon=False, ncols=3, color_map="magma_r", size=5, show=False)
    plt.savefig(f"Holloway_2021/figures/QC/{atribute}_sctk_multi_resolution_cluster_qc_{timestamp}.png", bbox_inches="tight")
    plt.show()

### Visualize QC metrics before filtering

In [ ]:
plt.figure(figsize=(15, 15), dpi=300)
sns.scatterplot(data=adata.obs, x='total_counts', y='n_genes_by_counts' , hue ='Source Name', alpha = 0.4, s=4)
plt.legend(title='sample', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(range(0, int(max(adata.obs['total_counts'])) + 1, 15000),rotation=90, fontsize = 10)
plt.yticks(range(0, int(max(adata.obs['n_genes_by_counts'])) + 1, 1000),fontsize = 10)
plt.title(f'Counts vs Genes - Before filtering')
plt.savefig(f"Holloway_2021/figures/QC/{atribute}_counts_vs_genes_before_filtering_{timestamp}.png", bbox_inches="tight")
plt.show()

In [ ]:
variables = 'n_counts', 'log1p_n_counts', 'log1p_n_genes', 'n_genes_by_counts', 'total_counts', 'percent_mito', 'percent_ribo', 'percent_hb', 'doublet_scores'

for var in variables:

    fig, ax = plt.subplots(figsize=(15, 10), ncols=2, gridspec_kw={'width_ratios': [4, 2]})

    sns.violinplot(data=adata.obs, x='Source Name', y=var, ax=ax[0])
   
    medians = adata.obs.groupby('Source Name')[var].median()
    
    ax[0].set_title(f'Violin Plot of {var} by Sample - Before filtering')
    ax[0].set_xlabel('Sample')
    ax[0].set_ylabel(var)
    ax[0].tick_params(axis='x', rotation=90)

    median_df = pd.DataFrame({'Sample': medians.index, 'Median': medians.values})

    ax[1].axis('off')
    ax[1].table(cellText=median_df.values, colLabels=median_df.columns, loc='center')
    ax[1].set_title('Median Values')
    
    plt.tight_layout()
    plt.savefig(f"Holloway_2021/figures/QC/{atribute}_{var}_before_filtering_{timestamp}.png", dpi = 300, bbox_inches="tight")
    plt.show()

In [ ]:
plt.figure(figsize=(10, 6), dpi=300)
sns.countplot(data=adata.obs, x='Source Name')
plt.ylim(0, 100000)
plt.xticks(rotation=90)
plt.title('Number of Cells per Sample - Before filtering')
plt.savefig(f"Holloway_2021/figures/QC/{atribute}_number_of_cells_before_filtering_{timestamp}.png", bbox_inches="tight")

### Filter cells that do not pass QC

In [7]:
adata_filtered = adata[adata.obs['consensus_passed_qc'] == True].copy()
adata_filtered

AnnData object with n_obs × n_vars = 10558 × 70711
    obs: 'sample_name', 'batch', 'barcode', 'Source Name', 'Comment[ENA_SAMPLE]', 'Comment[BioSD_SAMPLE]', 'Characteristics[organism]', 'Characteristics[disease]', 'Characteristics[genotype]', 'Characteristics[organism part]', 'Characteristics[cell type]', 'Characteristics[growth condition]', 'Characteristics[developmental stage]', 'Material Type', 'Protocol REF', 'Protocol REF.1', 'Protocol REF.2', 'Protocol REF.3', 'Extract Name', 'Comment[LIBRARY_LAYOUT]', 'Comment[LIBRARY_SELECTION]', 'Comment[LIBRARY_SOURCE]', 'Comment[LIBRARY_STRATEGY]', 'Comment[cdna read]', 'Comment[cdna read offset]', 'Comment[cdna read size]', 'Comment[cell barcode offset]', 'Comment[cell barcode read]', 'Comment[cell barcode size]', 'Comment[end bias]', 'Comment[input molecule]', 'Comment[library construction]', 'Comment[primer]', 'Comment[sample barcode offset]', 'Comment[sample barcode read]', 'Comment[sample barcode size]', 'Comment[single cell isolation]

### Visualize QC metrics after filtering

In [ ]:
plt.figure(figsize=(15, 15), dpi=300)
sns.scatterplot(data=adata_filtered.obs, x='total_counts', y='n_genes_by_counts' , hue ='Source Name', alpha = 0.4, s=4)
plt.legend(title='sample', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(range(0, int(max(adata_filtered.obs['total_counts'])) + 1, 3000),rotation=90, fontsize = 10)
plt.yticks(range(0, int(max(adata_filtered.obs['n_genes_by_counts'])) + 1, 1000),fontsize = 10)
plt.title(f'Counts vs Genes - After filtering')
plt.savefig(f"Holloway_2021/figures/QC/{atribute}_counts_vs_genes_after_filtering_{timestamp}.png", bbox_inches="tight")
plt.show()

In [ ]:
variables = 'n_counts', 'log1p_n_counts', 'log1p_n_genes', 'n_genes_by_counts', 'total_counts', 'percent_mito', 'percent_ribo', 'percent_hb', 'doublet_scores'

for var in variables:

    fig, ax = plt.subplots(figsize=(15, 10), ncols=2, gridspec_kw={'width_ratios': [4, 1]})

    sns.violinplot(data=adata_filtered.obs, x='Source Name', y=var, ax=ax[0])
   
    medians = adata_filtered.obs.groupby('Source Name')[var].median()
    
    ax[0].set_title(f'Violin Plot of {var} by Sample - After filtering')
    ax[0].set_xlabel('Sample')
    ax[0].set_ylabel(var)
    ax[0].tick_params(axis='x', rotation=90)

    median_df = pd.DataFrame({'Sample': medians.index, 'Median': medians.values})

    ax[1].axis('off')
    ax[1].table(cellText=median_df.values, colLabels=median_df.columns, loc='center')
    ax[1].set_title('Median Values')
    
    plt.tight_layout()
    plt.savefig(f"Holloway_2021/figures/QC/{atribute}_{var}_after_filtering_{timestamp}.png", dpi = 300, bbox_inches="tight")
    plt.show()

In [ ]:
plt.figure(figsize=(10, 6), dpi=300)
sns.countplot(data=adata_filtered.obs, x='Source Name')
plt.ylim(0, 100000)
plt.xticks(rotation=90)
plt.title('Number of Cells per Sample - After filtering')
plt.savefig(f"Holloway_2021/figures/QC/{atribute}_number_of_cells_after_filtering_{timestamp}.png", bbox_inches="tight")

## Sex covariate analysis

In [8]:
annot = sc.queries.biomart_annotations(
        "hsapiens",
        ["ensembl_gene_id", "external_gene_name", "start_position", "end_position", "chromosome_name"],
    ).set_index("external_gene_name")

* Chr Y genes calculation

In [9]:
chrY_genes = adata_filtered.var_names.intersection(annot.index[annot.chromosome_name == "Y"])

In [10]:
adata_filtered.obs['percent_chrY'] = np.sum(
    adata_filtered[:, chrY_genes].X, axis = 1) / np.sum(adata_filtered.X, axis = 1) * 100

+ XIST expression analysis

In [11]:
xist_counts = adata_filtered.X[:, adata_filtered.var_names.str.match('XIST')].toarray()

In [12]:
xist_percentage = (np.sum(xist_counts, axis=1) / adata_filtered.obs['total_counts']) * 100

In [13]:
xist_counts_series = pd.Series(xist_counts.squeeze(), index = adata_filtered.obs_names, name = "XIST-counts")
xist_percentage_series = pd.Series(xist_percentage, index=adata_filtered.obs_names, name="percent_XIST")

adata_filtered.obs["XIST-counts"] = xist_counts_series
adata_filtered.obs["XIST-percentage"] = xist_percentage_series

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(15, 15))
    sc.pl.violin(adata_filtered, ["XIST-counts", "percent_chrY"], jitter = 0.4, groupby = 'Source Name', rotation = 90, show=False)
    plt.savefig(f"Holloway_2021/figures/QC/{atribute}_sex_covariates_{timestamp}.png", bbox_inches="tight")
    plt.show()

In [15]:
average_xist_counts = adata_filtered.obs.groupby('Source Name')['XIST-counts'].mean()

In [16]:
average_percent_chrY = adata_filtered.obs.groupby('Source Name')['percent_chrY'].mean()

In [17]:
scaled_xist_counts = (average_xist_counts - average_xist_counts.min()) / (average_xist_counts.max() - average_xist_counts.min())
scaled_percent_chrY = (average_percent_chrY - average_percent_chrY.min()) / (average_percent_chrY.max() - average_percent_chrY.min())

In [18]:
sample_sex = pd.DataFrame({
    'scaled_XIST_counts': scaled_xist_counts,
    'scaled_percent_chrY': scaled_percent_chrY
})
sample_sex['sex'] = np.where(sample_sex['scaled_XIST_counts'] > sample_sex['scaled_percent_chrY'], 'female', 'male')

sample_sex

,scaled_XIST_counts,scaled_percent_chrY,sex
Source Name,,,
HT071-LWRN-EGF,0.000000,0.936146,male
HT071-LWRN-NRG1,0.000000,1.000000,male
HT323-LWRN-EGF,0.723792,0.000000,female
HT323-LWRN-EGF-NRG1,1.000000,0.000273,female
HT323-LWRN-NRG1,0.615690,0.000330,female


In [19]:
adata_filtered.obs = adata_filtered.obs.merge(sample_sex[['sex']], left_on='Source Name', right_index=True, how='left')
adata_filtered

AnnData object with n_obs × n_vars = 10558 × 70711
    obs: 'sample_name', 'batch', 'barcode', 'Source Name', 'Comment[ENA_SAMPLE]', 'Comment[BioSD_SAMPLE]', 'Characteristics[organism]', 'Characteristics[disease]', 'Characteristics[genotype]', 'Characteristics[organism part]', 'Characteristics[cell type]', 'Characteristics[growth condition]', 'Characteristics[developmental stage]', 'Material Type', 'Protocol REF', 'Protocol REF.1', 'Protocol REF.2', 'Protocol REF.3', 'Extract Name', 'Comment[LIBRARY_LAYOUT]', 'Comment[LIBRARY_SELECTION]', 'Comment[LIBRARY_SOURCE]', 'Comment[LIBRARY_STRATEGY]', 'Comment[cdna read]', 'Comment[cdna read offset]', 'Comment[cdna read size]', 'Comment[cell barcode offset]', 'Comment[cell barcode read]', 'Comment[cell barcode size]', 'Comment[end bias]', 'Comment[input molecule]', 'Comment[library construction]', 'Comment[primer]', 'Comment[sample barcode offset]', 'Comment[sample barcode read]', 'Comment[sample barcode size]', 'Comment[single cell isolation]

## Calculate cell cycle score

In [ ]:
cell_cycle_genes = pd.read_csv('regev_lab_cell_cycle_genes.txt', header=None)

In [21]:
s_genes = cell_cycle_genes.iloc[0:43]
g2m_genes = cell_cycle_genes.iloc[43:]

In [22]:
len(s_genes), len(g2m_genes)

(43, 54)

In [23]:
s_genes_in_adata = [gene for gene in s_genes[0] if gene in adata_filtered.var_names]
g2m_genes_in_adata = [gene for gene in g2m_genes[0] if gene in adata_filtered.var_names]
len(s_genes_in_adata), len(g2m_genes_in_adata)

(42, 52)

In [24]:
adata_log = ad.AnnData(X = adata_filtered.X,  var = adata_filtered.var, obs = adata_filtered.obs)
sc.pp.normalize_total(adata_log, target_sum = 1e6, exclude_highly_expressed = True)
sc.pp.log1p(adata_log)

In [25]:
sc.tl.score_genes_cell_cycle(adata_log, s_genes = s_genes_in_adata, g2m_genes = g2m_genes_in_adata)

In [26]:
adata_filtered.obs['S_score'] = adata_log.obs['S_score']
adata_filtered.obs['G2M_score'] = adata_log.obs['G2M_score']
adata_filtered.obs['Cell_cycle_phase'] = adata_log.obs['phase']

In [ ]:
plt.figure(figsize=(10, 6), dpi=300)
sns.countplot(data=adata_filtered.obs, x='Cell_cycle_phase')
plt.title('Number of Cells per Cell Cycle Phase')
plt.xlabel('Cell Cycle Phase')
plt.ylabel('Number of Cells')
plt.savefig(f"Holloway_2021/figures/QC/{atribute}_cell_cycle_phase_{timestamp}.png", bbox_inches="tight")
plt.show()

### Export anndata object

In [27]:
adata_filtered.uns['processing_history']

array(['{"step": "create raw anndata after mapping, no filtering", "timestamp": "06112024_121025"}',
       '{"timestamp": "27112024_160858", "step": "added qc metrics (generated with sctk), doublets info, filtered cells with less than 100 genes"}'],
      dtype=object)

In [29]:
current_history = adata_filtered.uns['processing_history'].tolist()

new_entry = json.dumps({
    'timestamp': timestamp,
    'step': 'filtered cells based on consensus qc generated with sctk (default parameters), added sex covariates and cell cycle phase',
})
current_history.append(new_entry)

adata_filtered.uns['processing_history'] = current_history

In [30]:
adata_filtered.uns['processing_history']

['{"step": "create raw anndata after mapping, no filtering", "timestamp": "06112024_121025"}',
 '{"timestamp": "27112024_160858", "step": "added qc metrics (generated with sctk), doublets info, filtered cells with less than 100 genes"}',
 '{"timestamp": "09122024_103213", "step": "filtered cells based on consensus qc generated with sctk (default parameters), added sex covariates and cell cycle phase"}']

In [31]:
adata_filtered

AnnData object with n_obs × n_vars = 10558 × 70711
    obs: 'sample_name', 'batch', 'barcode', 'Source Name', 'Comment[ENA_SAMPLE]', 'Comment[BioSD_SAMPLE]', 'Characteristics[organism]', 'Characteristics[disease]', 'Characteristics[genotype]', 'Characteristics[organism part]', 'Characteristics[cell type]', 'Characteristics[growth condition]', 'Characteristics[developmental stage]', 'Material Type', 'Protocol REF', 'Protocol REF.1', 'Protocol REF.2', 'Protocol REF.3', 'Extract Name', 'Comment[LIBRARY_LAYOUT]', 'Comment[LIBRARY_SELECTION]', 'Comment[LIBRARY_SOURCE]', 'Comment[LIBRARY_STRATEGY]', 'Comment[cdna read]', 'Comment[cdna read offset]', 'Comment[cdna read size]', 'Comment[cell barcode offset]', 'Comment[cell barcode read]', 'Comment[cell barcode size]', 'Comment[end bias]', 'Comment[input molecule]', 'Comment[library construction]', 'Comment[primer]', 'Comment[sample barcode offset]', 'Comment[sample barcode read]', 'Comment[sample barcode size]', 'Comment[single cell isolation]

In [ ]:
adata_filtered.write_h5ad(f"Holloway_2021/{project}_{species}_{atribute}_QC_filtered_{name}_{timestamp}_{counts}.h5ad")